In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
TITLE = dbutils.widgets.get("title")

BASE_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/raw"
SOURCE_PATH = f"{BASE_PATH}/{TITLE}.csv"

PREPARED_PATH = f"{BASE_PATH}/prepared"
LANDING_PATH = f"{BASE_PATH}/landing"

In [0]:
source_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", "true")
        .option("mode", "PERMISSIVE")
        .load(SOURCE_PATH)
)

In [0]:
window_spec = Window.orderBy(F.monotonically_increasing_id())
numbered_df = (
    source_df
    .withColumn(
        "_row_number", F.row_number().over(window_spec)
    )
)

In [0]:
batch1_df = (
    numbered_df
    .filter(F.col("_row_number").between(1,3000)).drop("_row_number")
)

batch2_df = (
    numbered_df
    .filter(F.col("_row_number").between(3001,6000)).drop("_row_number")
)

batch3_df = (
    numbered_df
    .filter(F.col("_row_number")>6000).drop("_row_number")
)

In [0]:
batch1_final = batch1_df

In [0]:
update_row = batch1_df.limit(1)
updated_row = (
    update_row
    .withColumn("title", F.concat(F.col("title"), F.lit("- UPDATED")))
)


In [0]:
duplicate_row = batch2_df.limit(1)

In [0]:
batch2_final = (
   batch2_df
   .unionByName(updated_row)
   .unionByName(duplicate_row) 
)

In [0]:
batch3_final = (
    batch3_df
    .withColumn(
        "imdb_rating", F.round(F.rand()*4 + 6, 1)
    )
)

In [0]:
(
    batch1_final
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{PREPARED_PATH}/batch_1")
)

(
    batch2_final
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{PREPARED_PATH}/batch_2")
)

(
    batch3_final
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{PREPARED_PATH}/batch_3")
)